# land surface temperature (LST) of bavaria from landsat 8 dataset from 2014-2025

* Landsat ST_B10 data is already LST in Kelvin after applying linear calibration factors

What this script does:
1) Read the CSV -> DataFrame with columns ['date', 'temperature']
2) Create a third column 'temperature_interp' using time-aware interpolation (no extrapolation)
3) Plot #1: date vs temperature, colored by recency using a 'summer' colormap (older=green, newer=yellow)
4) Plot #2: overlay each year's temperature vs calendar day, colored by year using 'summer' (older years=green, newer=yellow)
5) Plot #3: polar plot — angle is calendar day (Jan 1 at 12 o'clock, clockwise); radius = temperature with origin at −50 °C

All plots are made with Plotly and use the plotly dark theme.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import matplotlib.cm as cm
import matplotlib.colors as mcolors

In [ ]:
def load_and_prepare_csv(path: str) -> pd.DataFrame:
    """Load CSV, normalize to columns ['date','temperature'], and compute time-aware interpolation."""
    df_raw = pd.read_csv(path)

    # Normalize to two columns: date + temperature
    if df_raw.shape[1] == 2:
        df = df_raw.copy()
        df.columns = ["date", "temperature"]
    else:
        # Try to detect columns by name; otherwise fallback to first two
        date_candidates = [c for c in df_raw.columns if "date" in c.lower() or "time" in c.lower()]
        temp_candidates = [c for c in df_raw.columns if any(k in c.lower() for k in ["temp", "mean", "avg", "value"])]
        date_col = date_candidates[0] if date_candidates else df_raw.columns[0]
        if temp_candidates:
            temp_col = temp_candidates[0]
        else:
            temp_col = df_raw.columns[1] if df_raw.shape[1] > 1 else df_raw.columns[0]
        df = df_raw[[date_col, temp_col]].copy()
        df.columns = ["date", "temperature"]

    # Parse types
    df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=False)
    df["temperature"] = pd.to_numeric(df["temperature"], errors="coerce")

    # Clean and sort
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

    # Time-aware interpolation on the existing date index; no extrapolation beyond min/max
    df_interp = (
        df.set_index("date")["temperature"]
          .interpolate(method="time", limit_area="inside")
    )
    df["temperature_interp"] = df_interp.values

    # Convenience columns
    df["year"] = df["date"].dt.year
    df["day_of_year"] = df["date"].dt.dayofyear
    df["is_leap"] = df["date"].dt.is_leap_year
    df["days_in_year"] = np.where(df["is_leap"], 366, 365)

    return df


def matplotlib_summer_colors(n: int = 256):
    """Sample the Matplotlib 'summer' colormap and return hex colors for Plotly."""
    
    cmap = cm.get_cmap("summer", n)
    return [mcolors.to_hex(cmap(i)) for i in range(n)]


def colorscale_from_hex_list(hex_list):
    """Convert a list of hex colors to a Plotly continuous colorscale [[pos,color], ...]."""
    n = len(hex_list)
    return [[i / (n - 1), c] for i, c in enumerate(hex_list)]


def color_for_years(years: np.ndarray):
    """Map years -> hex color using 'summer' (older=green, newer=yellow)."""
    y_min, y_max = int(years.min()), int(years.max())
    palette = matplotlib_summer_colors(256)

    def to_color(y: int) -> str:
        if y_max == y_min:
            return palette[0]
        t = (y - y_min) / (y_max - y_min)
        idx = int(np.clip(round(t * 255), 0, 255))
        return palette[idx]

    return {int(y): to_color(int(y)) for y in np.unique(years)}


def date_to_scalar(dts: pd.Series) -> np.ndarray:
    """Normalize datetimes to [0,1] for recency coloring (older=0, newer=1)."""
    s = pd.to_datetime(dts).astype("int64")  # nanoseconds since epoch
    dmin, dmax = s.min(), s.max()
    if dmax == dmin:
        return np.zeros(len(s))
    return (s - dmin) / (dmax - dmin)


# --- Plots --------------------------------------------------------------------



In [ ]:
#mount google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path="trainingdata/ee-bay-avg-chart.csv"
df = load_and_prepare_csv(path)

In [ ]:
# plot1
y = df["temperature"].where(~df["temperature"].isna(), df["temperature_interp"])
recency = date_to_scalar(df["date"])
summer_scale = colorscale_from_hex_list(matplotlib_summer_colors(256))

fig = px.scatter(
        df,
        x="date",
        y=y,
        color=recency,
        color_continuous_scale=summer_scale,
        labels={"x": "Date", "y": "Temperature (°C)", "color": "Recency"},
        title="Average Temperature vs Date (older→green, newer→yellow)",
        template="plotly_dark",
)
fig.update_traces(
        mode="markers",
        marker=dict(size=2),
        hovertemplate="Date=%{x|%Y-%m-%d}<br>T=%{y:.2f}°C<extra></extra>",
)
fig.update_layout(coloraxis_colorbar=dict(tickvals=[0, 1], ticktext=["2014", "2025"]))

In [ ]:
# plot2

year_colors = color_for_years(df["year"].values)
fig = go.Figure()
for yr, g in df.groupby("year", sort=True):
    g = g.sort_values("day_of_year")
    y = g["temperature_interp"]
    if y.isna().all():
        continue
    fig.add_trace(go.Scatter(
        x=g["day_of_year"],
        y=y,
        mode="lines",
        name=str(yr),
        line=dict(width=2, color=year_colors[int(yr)]),
        opacity=0.45,
        hovertemplate=f"Year={yr}<br>Day=%{{x}}<br>T=%{{y:.2f}}°C<extra></extra>",
))
fig.update_layout(
    title="Temperature by Calendar Day (curves by year; older→green, newer→yellow)",
    xaxis_title="Calendar Day of Year",
    yaxis_title="Temperature (°C)",
    template="plotly_dark",
    legend_title_text="Year",
)

In [ ]:
# plot3
year_colors = color_for_years(df["year"].values)
fig = go.Figure()
for yr, g in df.groupby("year", sort=True):
    g = g.sort_values("date")
    theta = 360.0 * (g["day_of_year"] - 1) / g["days_in_year"]  # degrees
    r = g["temperature_interp"] + 50  # shift so −50°C is origin
    if r.isna().all():
        continue
    fig.add_trace(go.Scatterpolar(
        theta=theta,
        r=r,
        mode="lines",
        name=str(yr),
        line=dict(width=2, color=year_colors[int(yr)]),
        hovertemplate=(
            f"Year={yr}<br>Angle=%{{theta:.0f}}°"
            "<br>T=%{customdata:.2f}°C<extra></extra>"
        ),
            customdata=g["temperature_interp"],
    ))

fig.update_layout(
    title="Polar Temperature Cycle (Jan 1 at 12 o'clock; r origin = −50°C)",
    template="plotly_dark",
    polar=dict(
        angularaxis=dict(
            direction="clockwise",
            rotation=90,  # 0° at top (12 o'clock)
            tickmode="array",
            tickvals=[0, 90, 180, 270],
            ticktext=["0°", "90°", "180°", "270°"],
        ),
        radialaxis=dict(
            title="Temperature (°C)",
            tickmode="array",
            # Tick values expressed in shifted r, labels show true °C by subtracting 50
            tickvals=[0, 10, 20, 30, 40, 50, 60, 80, 100],
            ticktext=[f"{v - 50:.0f}" for v in [0, 10, 20, 30, 40, 50, 60, 80, 100]],
            showline=True,
        ),
    ),
    legend_title_text="Year",
)

In [ ]:
# plot4 - wip: requires some improvements the iso perspective is not useful, it needs some orthographic perspective when eliminating the z-Axis
# Colors per year (older -> green, newer -> yellow)
year_colors = color_for_years(df["year"].values)

# Time baseline and z units (days since first observation)
t0 = df["date"].min()

fig = go.Figure()

for yr, g in df.groupby("year", sort=True):
    g = g.sort_values("date")

    # Interpolated temps for continuity
    T = g["temperature_interp"]
    if T.isna().all():
        continue

    # Cylindrical -> Cartesian
    # Angle: day_of_year mapped to [0, 360), Jan 1 at 12 o'clock (rotation=90) and clockwise
    theta_deg = 360.0 * (g["day_of_year"] - 1) / g["days_in_year"]
    theta_rad = np.deg2rad(theta_deg)
    r = T + 50.0  # -50°C at origin

    x = r * np.cos(theta_rad)
    y = r * np.sin(theta_rad)

    # z = days since first observation (3D axes are linear; we label ticks with years later)
    z = (g["date"] - t0).dt.total_seconds() / 86400.0

    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode="lines",
        name=str(yr),
        line=dict(width=4, color=year_colors[int(yr)]),
        opacity=0.65,  # slight transparency to help overlapping years
        hovertemplate=(
            f"Year={yr}"
            "<br>θ=%{customdata[0]:.0f}°"
            "<br>T=%{customdata[1]:.2f}°C"
            "<br>z(day)=%{z:.1f}<extra></extra>"
        ),
        customdata=np.column_stack([theta_deg, T]),
    ))

# Build z-axis ticks labeled by year (use Jan 1 of each year if within range)
years = np.sort(df["year"].unique())
z_tickvals, z_ticktext = [], []
for y in years:
    jan1 = pd.Timestamp(year=y, month=1, day=1)
    if jan1 >= t0 and jan1 <= df["date"].max():
        z_tickvals.append((jan1 - t0).total_seconds() / 86400.0)
        z_ticktext.append(str(y))

fig.update_layout(
    title="3D Time-Polar Temperature (helix: angle=day, radius=T+50, z=time)",
    template="plotly_dark",
    scene=dict(
        xaxis_title="x = (T+50)·cos(θ)",
        yaxis_title="y = (T+50)·sin(θ)",
        zaxis_title="Time (days since first record)",
        zaxis=dict(tickmode="array", tickvals=z_tickvals, ticktext=z_ticktext, showgrid=True),
        aspectmode="data",  # preserve scale; the helix won't look squashed
    ),
    legend_title_text="Year",
)
